# Klasik Görüntü İşleme ile Mikroskobik Görüntülerde Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Öğrenci No:** 254312024 · **Seed:** 12024 + 2026 = **14050** · **Veri seti:** Cell Tracking Challenge – Fluo-N2DH-GOWT1 (training)

Bu notebook ödevin 2–12. bölümlerini sırasıyla ve tekrarlanabilir biçimde çalıştırır:

| Bölüm | İçerik | Çıktı |
|---|---|---|
| 2 | Veri setinin ön incelemesi | `figures/private/fig0_*.png` |
| 3 | Seed ile kare seçimi | `selected_frames.csv` |
| 4 | Geliştirme/doğrulama referans sayıları | `reference_counts_devval.csv` |
| 5 | Baseline (Otsu + CCL) | `experiments.csv` (B01–B06) |
| 6–10 | Kontrollü deneyler | `experiments.csv` (E01–…) |
| 10–11 | 3 aday pipeline + doğrulama | `validation_results.csv` |
| 12 | Nihai pipeline'ın kilitlenmesi, test tahmini | `final_pipeline.json`, `test_predictions_locked.csv`, `test_lock.json` |

> **Test güvenliği:** Bu notebook test referanslarını (`02_GT`) **hiçbir yerde okumaz**. Test değerlendirmesi, kilit dosyaları GitHub'a ayrı bir commit ile yüklendikten sonra `02_test_evaluation.ipynb` içinde yapılır.

## 0. Kurulum ve yollar

In [ ]:
import sys, json, time, hashlib, platform
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import scipy, skimage, matplotlib
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from cellcount import SEED, STUDENT_ID
from cellcount.io_utils import ensure_dataset, list_frames, load_image, reference_count, read_tif, tra_path
from cellcount.selection import select_frames, check_selection
from cellcount.pipeline import run_pipeline, evaluate_config, baseline_config, make_config, describe, save_config
from cellcount.metrics import count_metrics
from cellcount import experiments as X
from cellcount import viz

# Veri seti konumu: <proje>/data/Fluo-N2DH-GOWT1  (yoksa zip otomatik açılır)
DATA_DIR = PROJECT_DIR / "data"
ZIP_CANDIDATES = [DATA_DIR / "Fluo-N2DH-GOWT1.zip", PROJECT_DIR / "Fluo-N2DH-GOWT1.zip", PROJECT_DIR.parent / "Fluo-N2DH-GOWT1.zip"]
zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
ROOT = ensure_dataset(DATA_DIR, zip_path)

FIG_DIR = PROJECT_DIR / "figures"            # veri görüntüsü içermeyen grafikler (GitHub'a yüklenebilir)
PRIV_DIR = FIG_DIR / "private"               # veri görüntüsü içeren şekiller (yalnızca rapor; .gitignore)
FIG_DIR.mkdir(exist_ok=True); PRIV_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200, "font.size": 9})

print("Seed:", SEED, "| Python", platform.python_version(), "| OpenCV", cv2.__version__, "| NumPy", np.__version__,
      "| SciPy", scipy.__version__, "| scikit-image", skimage.__version__, "| pandas", pd.__version__)
print("Veri:", ROOT)

## 2. Veri setinin ön incelemesi
Sequence 01 ve 02'nin boyut, veri tipi, yoğunluk ve histogram özellikleri incelenir. (Bu bölümde yalnızca **ham görüntüler** kullanılır; hücre bazlı parlaklık analizi yalnızca Sequence 01 referanslarıyla yapılır.)

In [ ]:
frames01, frames02 = list_frames(ROOT, "01"), list_frames(ROOT, "02")
print("Sequence 01:", len(frames01), "kare", frames01[0], "-", frames01[-1])
print("Sequence 02:", len(frames02), "kare", frames02[0], "-", frames02[-1])

stats = []
for seq, frames in (("01", frames01), ("02", frames02)):
    for f in frames[::10]:
        im = load_image(ROOT, seq, f)
        p = np.percentile(im, [1, 50, 99, 99.9])
        stats.append(dict(sequence=seq, frame=f, shape=im.shape, dtype=str(im.dtype), min=int(im.min()), max=int(im.max()),
                          mean=round(float(im.mean()), 2), std=round(float(im.std()), 2), p50=p[1], p99=p[2], p99_9=p[3],
                          zero_ratio=round(float((im == 0).mean()), 3)))
stats = pd.DataFrame(stats)
stats.to_csv("dataset_stats.csv", index=False)
stats.groupby("sequence")[["mean", "std", "max", "p99", "p99_9", "zero_ratio"]].mean().round(2)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for i, (seq, f) in enumerate((("01", frames01[0]), ("02", frames02[0]))):
    im = load_image(ROOT, seq, f)
    ax[i, 0].imshow(im, cmap="gray"); ax[i, 0].set_title(f"Seq {seq} t{f:03d} (ham, 0-255)")
    ax[i, 1].imshow(viz.display_gain(im), cmap="gray"); ax[i, 1].set_title("Gösterim için kazançlı (p99.5)")
    ax[i, 2].hist(im.ravel(), bins=256, range=(0, 255), log=True); ax[i, 2].set_title(f"Histogram (log) – Seq {seq}")
for a in ax[:, :2].ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig0_dataset_overview.png", bbox_inches="tight"); plt.show()

In [ ]:
# Sequence 01: referans (TRA) hücrelerinin alan ve parlaklık dağılımı -> hücreler arası farklılık
rows = []
for f in frames01[::10]:
    im, tra = load_image(ROOT, "01", f), read_tif(tra_path(ROOT, "01", f))
    for lab in np.unique(tra)[1:]:
        px = im[tra == lab]
        rows.append(dict(frame=f, cell=int(lab), marker_area=int(px.size), median_intensity=float(np.median(px))))
cells = pd.DataFrame(rows)
print(cells[["marker_area", "median_intensity"]].describe().round(1))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(cells.median_intensity, bins=40); ax[0].set_xlabel("Hücre medyan yoğunluğu"); ax[0].set_ylabel("Hücre sayısı")
ax[1].hist(cells.marker_area, bins=40); ax[1].set_xlabel("TRA işaretçi alanı (px)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig0_cell_intensity_area_distribution.png", bbox_inches="tight"); plt.show()

In [ ]:
# Şekil ve boyut farklılıkları: Sequence 01 piksel düzeyinde SEG anotasyonları (01_GT/SEG).
# Not: GOWT1'de SEG anotasyonu yalnızca bazı karelerde ve karelerdeki hücrelerin bir kısmı için verilmiştir.
from skimage.measure import regionprops_table
shape_rows = []
for p in sorted((ROOT / "01_GT" / "SEG").glob("man_seg*.tif")):
    f = int(p.stem[-3:]); seg = read_tif(p)
    t = pd.DataFrame(regionprops_table(seg, intensity_image=load_image(ROOT, "01", f),
                                       properties=("label", "area", "equivalent_diameter_area", "eccentricity", "solidity", "intensity_mean")))
    t["frame"] = f; shape_rows.append(t)
shapes = pd.concat(shape_rows, ignore_index=True)
shapes.to_csv("cell_shape_stats.csv", index=False)
print(f"{len(shapes)} anotasyonlu çekirdek, {shapes.frame.nunique()} kare")
print(shapes[["area", "equivalent_diameter_area", "eccentricity", "solidity", "intensity_mean"]].describe().round(2))
fig, ax = plt.subplots(1, 4, figsize=(15, 3))
for a, (c, lbl) in zip(ax, [("area", "Alan (px)"), ("equivalent_diameter_area", "Eşdeğer çap (px)"), ("eccentricity", "Eksantriklik"), ("solidity", "Katılık (solidity)")]):
    a.hist(shapes[c], bins=30); a.set_xlabel(lbl)
ax[0].set_ylabel("Çekirdek sayısı")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig0_cell_shape_distribution.png", bbox_inches="tight"); plt.show()

**Gözlemler (rapora aktarılacak):**
* Görüntüler 1024×1024, 8-bit (`uint8`), LZW sıkıştırmalı TIFF. Piksellerin büyük bölümü 0 (koyu arka plan); histogram sağa çarpık ve seyrek (kuantizasyon boşlukları).
* Hücre parlaklıkları çok heterojendir: bazı çekirdeklerin medyan yoğunluğu 100'ün üzerindeyken bazıları arka plana çok yakındır (≈3–10). Bu, tek bir global eşiğin soluk hücreleri kaçırmasına neden olur.
* Sequence 02, Sequence 01'e göre belirgin biçimde daha koyudur (p99 ≈ 30 vs ≈ 70) → sabit (mutlak) parametrelerin genelleme riski.
* Çekirdek içi doku (nükleol), birbirine temas eden çekirdekler, mitoz/apoptoz kaynaklı küçük parlak yapılar ve görüntü kenarında kesilen hücreler diğer güçlüklerdir.

## 3. Geliştirme, doğrulama ve test karelerinin seçilmesi
`numpy.random.default_rng(14050)` ile deterministik seçim; kurallar `cellcount/selection.py` içinde açıklanmıştır. Seçim, algoritma geliştirilmeden önce yapılır ve sabittir.

In [ ]:
selected = select_frames(frames01, frames02, SEED)
check_selection(selected)
selected.to_csv("selected_frames.csv", index=False)
DEV = selected.query("group == 'development'").frame.tolist()
VAL = selected.query("group == 'validation'").frame.tolist()
TEST = selected.query("group == 'test'")[["frame", "test_block"]]
print("Development (01):", DEV)
print("Validation  (01):", VAL)
for b, g in TEST.groupby("test_block"): print(f"Test blok {b} (02):", g.frame.tolist())
selected

## 4. Referans hücre sayıları (yalnızca geliştirme + doğrulama)
`01_GT/TRA/man_trackXXX.tif` içinde 0 dışındaki farklı etiket kimlikleri sayılır. Test referansları bu notebook'ta **okunmaz**.

In [ ]:
ref_rows = [dict(group=g, sequence="01", frame=f, ref_count=reference_count(ROOT, "01", f))
            for g, frames in (("development", DEV), ("validation", VAL)) for f in frames]
refs_df = pd.DataFrame(ref_rows)
assert refs_df.ref_count.notna().all(), "Referansı olmayan kare var"
refs_df.to_csv("reference_counts_devval.csv", index=False)
REF = dict(zip(refs_df.frame, refs_df.ref_count))
dev_imgs = {f: load_image(ROOT, "01", f) for f in DEV}
val_imgs = {f: load_image(ROOT, "01", f) for f in VAL}
dev_refs = {f: REF[f] for f in DEV}; val_refs = {f: REF[f] for f in VAL}
refs_df.pivot_table(index="group", values="ref_count", aggfunc=["count", "mean", "min", "max"])

In [ ]:
# Görsel kontrol (Bölüm 4.5): TRA işaretçi merkezleri, geliştirme ve doğrulama karelerinin ham görüntüsü üzerinde
fig, ax = plt.subplots(3, 5, figsize=(18, 11.5))
for a, (g, f) in zip(ax.ravel(), [("dev", f) for f in DEV] + [("val", f) for f in VAL]):
    img = dev_imgs[f] if g == "dev" else val_imgs[f]
    a.imshow(viz.display_gain(img), cmap="gray"); viz.overlay_markers(a, read_tif(tra_path(ROOT, "01", f)))
    a.set_title(f"{g} t{f:03d} – referans {REF[f]}"); a.axis("off")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig_ref_check_devval.png", bbox_inches="tight"); plt.show()

*Not:* TRA anotasyonları hücre başına bir işaretçi içerir; görüntü kenarında kısmen görünen hücreler de sayılır. Referans sayısı ikili maskedeki bağlı bileşen sayısıyla aynı olmak zorunda değildir (temas eden hücreler tek bileşen oluşturabilir).

## 5. Baseline: Gri seviye → Otsu → Bağlı bileşen → Sayım
Tek serbest parametre olan **sabit minimum nesne alanı** yalnızca geliştirme kümesinde seçilir ve sonra dondurulur.

In [ ]:
baseline_rows, BASELINE = X.run_baseline_stage(dev_imgs, dev_refs)
pd.DataFrame(baseline_rows)[["exp_id", "changed", "dev_MAE", "dev_RMSE", "dev_Bias", "mean_time_ms", "dev_preds", "selected"]].round(3)

In [ ]:
print("Dondurulan baseline:", describe(BASELINE))
f = DEV[0]
cnt, lab, st = run_pipeline(dev_imgs[f], BASELINE, return_stages=True)
t_otsu = cv2.threshold(dev_imgs[f], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(viz.display_gain(dev_imgs[f]), cmap="gray"); ax[0].set_title(f"t{f:03d} (gösterim kazançlı) – referans {REF[f]}")
ax[1].imshow(st["mask"], cmap="gray"); ax[1].set_title(f"Otsu maskesi (T={t_otsu:.0f})")
ax[2].imshow(viz.overlay_labels(dev_imgs[f], lab)); ax[2].set_title(f"Baseline CCL, min_area={BASELINE['min_area']} → {cnt} nesne")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig_baseline_example.png", bbox_inches="tight"); plt.show()

## 6–10. Kontrollü deneyler (geliştirme kümesi, 10 kare)
Aşamalı protokol (`cellcount/experiments.py`): her aşamada **yalnızca bir bileşen** değiştirilir, diğerleri bir önceki aşamanın kazananında sabit tutulur.

S1 yoğunluk → S2 filtre → S3 segmentasyon → S4 morfoloji/işlem sırası → S5 temas eden hücre ayrıştırma → S6 min alan.

Başlangıç (referans) konfigürasyonu: `none → Gaussian 5×5 → Otsu → open(5)+fill_holes → ayrıştırma yok → min_area=400`.
Histogram Matching'in referans görüntüsü geliştirme kümesinin ilk karesidir.

In [ ]:
HM_REF_FRAME = DEV[0]
HM_REF = dev_imgs[HM_REF_FRAME]
t0 = time.time()
exp_rows, GREEDY_BEST = X.run_greedy_search(dev_imgs, dev_refs, HM_REF)
print(f"{len(exp_rows)} deney, {time.time() - t0:.1f} s")

In [ ]:
all_rows = baseline_rows + exp_rows
experiments = X.to_dataframe(all_rows)
cols = ["exp_id", "stage", "intensity", "filter", "segmentation", "morphology", "separation", "min_area",
        "dev_MAE", "dev_RMSE", "dev_Bias", "dev_ExactAcc", "mean_time_ms", "selected", "dev_preds"]
experiments = experiments[cols]
experiments.to_csv("experiments.csv", index=False)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 250)
experiments.round(3)

In [ ]:
# Parametre etkisi: aşama bazında geliştirme MAE'si (veri görüntüsü içermez)
fig, axes = plt.subplots(len(X.STAGES), 1, figsize=(10, 2.6 * len(X.STAGES)))
for ax, (s, key, _) in zip(axes, X.STAGES):
    d = experiments[experiments.stage == s]
    labels = d[key].astype(str).str.replace(r",?p_high=99.5|,?min_peak=3|,?n_background=1|,?tile=8", "", regex=True)
    colors = ["tab:green" if v else "tab:blue" for v in d.selected]
    ax.barh(range(len(d)), d.dev_MAE, color=colors); ax.set_yticks(range(len(d))); ax.set_yticklabels([f"{e}: {l}" for e, l in zip(d.exp_id, labels)], fontsize=7)
    ax.invert_yaxis(); ax.set_xlabel("Geliştirme MAE"); ax.set_title(s); ax.set_xscale("symlog", linthresh=2)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_parameter_effects.png", bbox_inches="tight"); plt.show()

### Şekil 2 – Yoğunluk dönüşümleri ve histogramlar (aynı geliştirme karesi)

In [ ]:
from cellcount import ops
# Histogram Matching referansı DEV[0] olduğu için şekil, dönüşümün anlamlı görülebilmesi adına DEV[1] üzerinde çizilir.
f = DEV[1]; im = dev_imgs[f]
s1 = experiments[experiments.stage == "S1-intensity"]
shown = [{"method": "none"}, {"method": "stretch", "p_low": 1.0, "p_high": 99.5}, {"method": "gamma", "gamma": 0.3, "p_high": 99.5},
         {"method": "hist_eq"}, {"method": "clahe", "clip_limit": 2.0, "tile": 8}, {"method": "hist_match"}]
fig, ax = plt.subplots(3, len(shown), figsize=(3.2 * len(shown), 9.8), gridspec_kw={"height_ratios": [3, 1.2, 3]})
for j, it in enumerate(shown):
    out = ops.apply_intensity(im, it, HM_REF)
    name = describe(make_config(intensity=it))["intensity"]
    mae = s1.loc[s1.intensity == name, "dev_MAE"]
    viz.show_with_hist(ax[0, j], ax[1, j], out, f"{name}\n(dev MAE={mae.iloc[0]:.2f})" if len(mae) else name)
    # Aynı dönüşümün segmentasyon ve sayıma etkisi (referans konfigürasyonun geri kalanı sabit)
    cnt, lab, _ = run_pipeline(im, dict(X.REFERENCE, intensity=it), HM_REF)
    ax[2, j].imshow(viz.overlay_labels(im, lab)); ax[2, j].axis("off")
    ax[2, j].set_title(f"Gauss5→Otsu→open5+fill\nsayı={cnt} (referans {REF[f]})", fontsize=8)
plt.suptitle(f"Şekil 2. Orijinal ve yoğunluk dönüşümleri, histogramları ve segmentasyon sonuçları – dev t{f:03d} (HM referansı: dev t{HM_REF_FRAME:03d})")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig2_intensity_transforms.png", bbox_inches="tight"); plt.show()

# Ham görüntü düşük seviyelerde kuantizedir: histogram eşitlemenin bu seviyeleri nereye taşıdığı
he = ops.hist_equalize(im)
print("Ham → HE eşlemesi (ilk seviyeler):", {int(v): int(he[im == v][0]) for v in np.unique(im)[:6]})

### Filtre karşılaştırması (aynı kare, S1 kazananı üzerinde)

In [ ]:
s1_best = X.best_in_stage(exp_rows, "S1-intensity")["config"]
pre = ops.apply_intensity(im, s1_best["intensity"], HM_REF)
crop = (slice(350, 650), slice(250, 550))
s2 = experiments[experiments.stage == "S2-filter"]
fig, ax = plt.subplots(2, 5, figsize=(17, 8))
for a, opt, (_, r) in zip(ax.ravel(), X.FILTER_OPTIONS, s2.iterrows()):
    out = pre
    for flt in opt: out = ops.apply_filter(out, flt)
    a.imshow(out[crop], cmap="gray", vmin=0, vmax=255); a.set_title(f"{r.exp_id}: {r['filter']}\nMAE={r.dev_MAE:.2f}", fontsize=8); a.axis("off")
plt.suptitle(f"Filtrelerin etkisi (yakınlaştırılmış bölge, {s1_best['intensity']['method']} sonrası)")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig_filters.png", bbox_inches="tight"); plt.show()

### Şekil 3 – Segmentasyon ve morfolojik işlem çıktıları

In [ ]:
def stage_configs(stage):
    return [(r["exp_id"], r["config"], r["dev_MAE"]) for r in exp_rows if r["stage"] == stage]

pick_seg = ["fixed(threshold=160)", "otsu", "triangle", "adaptive(block_size=151,C=-5,floor=30)", "kmeans(k=3,n_background=1)"]
pick_morph = ["none", "erode(5) -> fill_holes", "dilate(5) -> fill_holes", "open(3) -> fill_holes", "open(15) -> fill_holes",
              "close(9) -> fill_holes", "open(9) -> close(9) -> fill_holes", "close(9) -> open(9) -> fill_holes"]
sel = [(e, c, m) for e, c, m in stage_configs("S3-segmentation") if describe(c)["segmentation"] in pick_seg] + \
      [(e, c, m) for e, c, m in stage_configs("S4-morphology") if describe(c)["morphology"] in pick_morph]
ncol = 5; nrow = int(np.ceil(len(sel) / ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.8 * nrow))
for a in ax.ravel(): a.axis("off")
for a, (e, c, m) in zip(ax.ravel(), sel):
    cnt, lab, _ = run_pipeline(im, c, HM_REF)
    d = describe(c)
    a.imshow(viz.overlay_labels(im, lab)); a.set_title(f"{e}: {d['segmentation']}\n{d['morphology']}\nsayı={cnt} (ref {REF[f]}), dev MAE={m:.2f}", fontsize=7)
plt.suptitle(f"Şekil 3. Segmentasyon (üst) ve morfoloji (alt) karşılaştırması – dev t{f:03d}")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig3_segmentation_morphology.png", bbox_inches="tight"); plt.show()

In [ ]:
# Temas eden hücrelerin ayrıştırılması: distance transform + watershed
s4_best = X.best_in_stage(exp_rows, "S4-morphology")["config"]
f_touch = max(DEV, key=lambda k: REF[k])  # en kalabalık geliştirme karesi
fig, ax = plt.subplots(1, 4, figsize=(18, 5))
cnt0, lab0, st0 = run_pipeline(dev_imgs[f_touch], s4_best, HM_REF, return_stages=True)
ax[0].imshow(viz.overlay_labels(dev_imgs[f_touch], lab0)); ax[0].set_title(f"Ayrıştırma yok → {cnt0} (ref {REF[f_touch]})")
for a, sep in zip(ax[1:], X.SEPARATION_OPTIONS[1:4]):
    c = dict(s4_best, separation=sep)
    cnt, lab, st = run_pipeline(dev_imgs[f_touch], c, HM_REF, return_stages=True)
    a.imshow(viz.overlay_labels(dev_imgs[f_touch], lab)); a.set_title(f"{describe(c)['separation']} → {cnt}", fontsize=8)
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig_watershed_separation.png", bbox_inches="tight"); plt.show()

## 10. Aday pipeline'lar
Deney sonuçlarından üç farklı tasarım felsefesine sahip tam pipeline oluşturulur:

* **Pipeline A – Basit iyileştirilmiş:** S1–S4 aşama kazananları (yoğunluk + filtre + Otsu + morfoloji), temas ayrıştırma **yok**.
* **Pipeline B – Tam aşamalı arama sonucu:** S1–S6 kazananları (A + distance-transform/watershed + min alan).
* **Pipeline C – Alternatif yoğunluk ailesi:** Histogram tabanlı dönüşüm yerine görüntüye özgü normalize edilmiş **gamma (γ=0.3)** nokta dönüşümü + Gaussian 9×9 + Otsu + open(5) + fill holes, min_area=200 (S1'in en iyi nokta-dönüşümü; daha düşük gürültü yükseltmesi, ancak çok soluk hücreleri kaçırma riski).

In [ ]:
A = dict(X.best_in_stage(exp_rows, "S5-separation", lambda r: r["config"]["separation"]["method"] == "none")["config"], name="Pipeline A")
B = dict(GREEDY_BEST, name="Pipeline B")
Cc = make_config(name="Pipeline C", intensity={"method": "gamma", "gamma": 0.3, "p_high": 99.5},
                 filter=[{"method": "gaussian", "ksize": 9}], segmentation={"method": "otsu"},
                 morphology=[{"op": "open", "ksize": 5}, {"op": "fill_holes"}], separation={"method": "none"}, min_area=200)
CANDIDATES = [A, B, Cc]

def complexity(cfg):
    return 1 + len(cfg["filter"]) + 1 + len(cfg["morphology"]) + (cfg["separation"]["method"] != "none") * 2 + (cfg["intensity"]["method"] != "none")

dev_rows = []
for cfg in [BASELINE] + CANDIDATES:
    m = X.evaluate(cfg, dev_imgs, dev_refs, HM_REF)
    dev_rows.append(dict(pipeline=cfg["name"], **describe(cfg), dev_MAE=m["MAE"], dev_RMSE=m["RMSE"], dev_Bias=m["Bias"], dev_preds=m["preds"]))
pd.DataFrame(dev_rows).round(3)

In [ ]:
# Tüm geliştirme görüntülerinin görsel incelemesi (Bölüm 10.6): baseline ve her aday pipeline için
# camgöbeği "+" işaretleri TRA referans hücrelerini gösterir.
for cfg in [BASELINE] + CANDIDATES:
    fig, ax = plt.subplots(2, 5, figsize=(20, 8.6))
    for a, f in zip(ax.ravel(), DEV):
        cnt, lab, _ = run_pipeline(dev_imgs[f], cfg, HM_REF)
        a.imshow(viz.overlay_labels(dev_imgs[f], lab)); viz.overlay_markers(a, read_tif(tra_path(ROOT, "01", f)), "cyan")
        a.set_title(f"t{f:03d}: tahmin {cnt} / referans {REF[f]} ({cnt - REF[f]:+d})", fontsize=9); a.axis("off")
    d = describe(cfg)
    plt.suptitle(f"{cfg['name']} – tüm geliştirme kareleri | {d['intensity']} → {d['filter']} → {d['segmentation']} → "
                 f"{d['morphology']} → {d['separation']} → min_area={d['min_area']}", fontsize=9)
    plt.tight_layout()
    slug = "".join(ch for ch in cfg["name"].lower() if ch.isalnum())
    plt.savefig(PRIV_DIR / f"fig_dev_all_{slug}.png", bbox_inches="tight"); plt.show()

## 11. Doğrulama (5 kare) ve nihai pipeline seçimi
Seçim kuralı: en düşük doğrulama MAE → eşitlikte RMSE → ortalama süre → karmaşıklık (adım sayısı).

In [ ]:
N_TIMING_REPEATS = 3  # süre ölçümünün kararlılığı için
val_rows = []
for cfg in [BASELINE] + CANDIDATES:
    per = evaluate_config(cfg, val_imgs, val_refs, HM_REF)
    times = [np.mean([evaluate_config(cfg, {f: val_imgs[f]}, None, HM_REF)[0]["time_s"] for _ in range(N_TIMING_REPEATS)]) for f in VAL]
    m = count_metrics([r["pred"] for r in per], [r["ref"] for r in per], times)
    val_rows.append(dict(pipeline=cfg["name"], is_candidate=cfg is not BASELINE, **describe(cfg),
                         val_MAE=m["MAE"], val_RMSE=m["RMSE"], val_Bias=m["Bias"], val_ExactAcc=m["ExactAcc"],
                         mean_time_ms=1000 * m["mean_time_s"], complexity=complexity(cfg),
                         val_preds=" ".join(str(r["pred"]) for r in per), val_refs=" ".join(str(r["ref"]) for r in per)))
validation = pd.DataFrame(val_rows)
# Geliştirme kümesi (10 kare) sonuçları da aynı tabloya eklenir (Bölüm 10.4)
validation = validation.merge(pd.DataFrame(dev_rows)[["pipeline", "dev_MAE", "dev_RMSE", "dev_Bias"]], on="pipeline", how="left")
cand = validation[validation.is_candidate].copy()
cand["_k"] = list(zip(cand.val_MAE.round(6), cand.val_RMSE.round(6), cand.mean_time_ms, cand.complexity))
winner = cand.sort_values("_k").iloc[0].pipeline
validation["selected"] = validation.pipeline == winner
validation.to_csv("validation_results.csv", index=False)
FINAL = next(c for c in CANDIDATES if c["name"] == winner)
print("Seçilen nihai pipeline:", winner)
validation[["pipeline", "intensity", "filter", "segmentation", "morphology", "separation", "min_area", "dev_MAE",
            "val_MAE", "val_RMSE", "val_Bias", "mean_time_ms", "complexity", "val_preds", "val_refs", "selected"]].round(3)

### Şekil 1 – Nihai pipeline akış diyagramı ve Şekil 4 – aşama aşama çıktılar

In [ ]:
viz.flowchart(viz.config_to_steps(FINAL), f"Şekil 1. Nihai pipeline ({FINAL['name']})", FIG_DIR / "fig1_final_pipeline_flowchart.png"); plt.show()

In [ ]:
f = VAL[0]; im = val_imgs[f]
cnt, lab, st = run_pipeline(im, FINAL, HM_REF, return_stages=True)
d = describe(FINAL)
panels = [("original", "Orijinal (gösterim kazançlı)", viz.display_gain(st["original"]), "gray"),
          ("intensity", f"Yoğunluk: {d['intensity']}", st["intensity"], "gray"),
          ("filtered", f"Filtre: {d['filter']}", st["filtered"], "gray"),
          ("mask", f"Segmentasyon: {d['segmentation']}", st["mask"], "gray"),
          ("morph", f"Morfoloji: {d['morphology']}", st["morph"], "gray")]
if "distance" in st: panels.append(("distance", "Distance transform", st["distance"], "magma"))
panels.append(("labels", f"Sonuç: {cnt} hücre (referans {REF[f]})\n{d['separation']}, min_area={d['min_area']}", viz.overlay_labels(im, st["labels"]), None))
fig, ax = plt.subplots(2, 4, figsize=(18, 9.5))
for a in ax.ravel(): a.axis("off")
for a, (_, t, img, cm) in zip(ax.ravel(), panels):
    a.imshow(img, cmap=cm); a.set_title(t, fontsize=8)
plt.suptitle(f"Şekil 4. Nihai pipeline aşamaları – doğrulama karesi t{f:03d}", y=1.01)
plt.tight_layout(); plt.savefig(PRIV_DIR / "fig4_final_pipeline_stages.png", bbox_inches="tight"); plt.show()

## 12. Nihai pipeline'ın kilitlenmesi ve test tahminlerinin kaydedilmesi
1. `final_pipeline.json` yazılır (nihai pipeline + dondurulmuş baseline + seçim gerekçesi).
2. 15 test karesinde **referans okunmadan** nihai ve baseline tahminleri yapılır → `test_predictions_locked.csv`.
3. Her iki dosyanın SHA-256 özeti `test_lock.json` içine yazılır.

> ⚠️ **Bu hücreden sonra durun ve GitHub'a ayrı bir commit atın:**
> `git add final_pipeline.json test_predictions_locked.csv test_lock.json && git commit -m "Lock final pipeline and test predictions"`
> Ardından `02_test_evaluation.ipynb` çalıştırılır.

In [ ]:
LOCK_FILES = [Path("final_pipeline.json"), Path("test_predictions_locked.csv"), Path("test_lock.json")]
ALREADY_LOCKED = all(p.exists() for p in LOCK_FILES)
OUT_JSON = Path("scratch_final_pipeline.json") if ALREADY_LOCKED else Path("final_pipeline.json")

sel_row = validation[validation.selected].iloc[0]
save_config(FINAL, OUT_JSON, extra={
    "student_id": STUDENT_ID, "seed": SEED,
    "histogram_matching_reference": {"sequence": "01", "frame": HM_REF_FRAME, "used": FINAL["intensity"]["method"] == "hist_match"},
    "baseline": BASELINE,
    "selection_rule": "min validation MAE -> RMSE -> mean time -> complexity",
    "validation": {k: (float(sel_row[k]) if k != "val_preds" else sel_row[k]) for k in ["val_MAE", "val_RMSE", "val_Bias", "mean_time_ms", "val_preds"]},
    "software": {"python": platform.python_version(), "opencv": cv2.__version__, "numpy": np.__version__,
                 "scipy": scipy.__version__, "scikit-image": skimage.__version__},
})

# --- Test tahmini: yalnızca ham görüntüler okunur (02_GT okunmaz) ---
pred_rows = []
for _, r in TEST.iterrows():
    img = load_image(ROOT, "02", int(r.frame))
    t0 = time.perf_counter(); nb, _, _ = run_pipeline(img, BASELINE); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); nf, _, _ = run_pipeline(img, FINAL, HM_REF); tf = time.perf_counter() - t0
    pred_rows.append(dict(sequence="02", frame=int(r.frame), test_block=int(r.test_block), file_name=f"02/t{int(r.frame):03d}.tif",
                          baseline_pred=nb, final_pred=nf, baseline_time_ms=round(1000 * tb, 2), final_time_ms=round(1000 * tf, 2)))
locked = pd.DataFrame(pred_rows)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

if ALREADY_LOCKED:
    # Kilit zaten mevcut: dosyalar DEĞİŞTİRİLMEZ, yalnızca tekrarlanabilirlik kontrol edilir.
    prev = pd.read_csv("test_predictions_locked.csv")
    same_cfg = json.load(open(OUT_JSON, encoding="utf-8"))["pipeline"] == json.load(open("final_pipeline.json", encoding="utf-8"))["pipeline"]
    same_pred = prev[["frame", "baseline_pred", "final_pred"]].equals(locked[["frame", "baseline_pred", "final_pred"]])
    OUT_JSON.unlink()
    print("Kilit dosyaları mevcut; üzerine yazılmadı.")
    print("Aynı nihai konfigürasyon:", same_cfg, "| Aynı test tahminleri:", same_pred)
    assert same_cfg and same_pred, "Yeniden üretilen sonuçlar kilitli sonuçlarla uyuşmuyor!"
else:
    locked.to_csv("test_predictions_locked.csv", index=False)
    json.dump({"final_pipeline.json": sha("final_pipeline.json"), "test_predictions_locked.csv": sha("test_predictions_locked.csv"),
               "locked_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open("test_lock.json", "w"), indent=2)
print(open("test_lock.json").read())
locked

**Bu noktada kilitleme tamamlandı.** Test sonuçları incelendikten sonra algoritma, parametreler veya tahminler değiştirilmeyecektir.

GitHub'a yüklemeden önce notebook çıktılarını temizleyin (veri seti görüntüleri gömülü olmamalı):
`jupyter nbconvert --clear-output --inplace main.ipynb`